# Oxford dataset初步处理

### Oxford batteryclass 仅包含CC段数据

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from scipy.io import loadmat
from scipy import interpolate
import pandas as pd
from scipy.signal import savgol_filter

: 

In [ ]:
#数据处理用hampel利用局部中值代替异常值
def hampel_filter(x, window_size=7, n_sigma=3):
    x = x.copy()

    k = 1.4826

    for i in range(len(x)):
        start = max(i - window_size, 0)
        end = min(i + window_size + 1, len(x))

        window = x[start:end]
        median = np.median(window)
        mad = k * np.median(np.abs(window - median))

        if mad == 0:
            continue
        if np.abs(x[i] - median) > n_sigma * mad:
            x[i] = median

    return x

In [ ]:
path = 'Oxford_Battery_Degradation_Dataset_1.mat'
mat = loadmat(path)

In [ ]:
cell_number = 'Cell2'
# cell_number可选‘Cell1’,'Cell2', ...'Cell8'
file_name = 'Oxford ' + cell_number

In [ ]:
class Battery:
    def __init__(self,cell_number):
        self.cell = mat[cell_number]
        self.cycle_data = self.cell[0,0]
        self.cycle_life = len(self.cycle_data)
        self.cycle_list = list(self.cell[0,0].dtype.names)
        self.q0 = self.cell['cyc0000'][0][0]['C1dc'][0][0]['q'][0][0][-1]

    def get_SOH(self,cycle):
        cycle_num = self.cycle_list[cycle]
        q_dc = self.cell[cycle_num][0][0]['C1dc'][0][0]['q'][0][0][-1]
        SOH = q_dc / self.q0
        return SOH[0]

    def get_value(self,cycle,variable):
        # 可选variable有：'t', 'v', 'q', 'T',只选C1ch的数据用于特征提取
        cycle_num = self.cycle_list[cycle]
        value = self.cell[cycle_num][0][0]['C1ch'][0][0][variable][0][0]
        return value

    def get_CC_value(self,cycle,variable,voltage_range):
        # 由于只包含CC段数据，voltage_range实际只输入一个值，为下边界值。上边界为最后一个值
        value = self.get_value(cycle=cycle, variable=variable)
        voltage = self.get_value(cycle=cycle, variable='v')
        index = np.where(voltage >= voltage_range) 
        value = value[index]
        return value
    
    def get_IC_curve(self,cycle,voltage_range=3.8,step_len=0.01):
        Q = self.get_CC_value(cycle=cycle,variable='q',voltage_range=voltage_range)
        V = self.get_CC_value(cycle=cycle,variable='v',voltage_range=voltage_range)

        Q = hampel_filter(Q, window_size=7, n_sigma=3)
        V = hampel_filter(V, window_size=7, n_sigma=3)
        Q = savgol_filter(Q, window_length=11, polyorder=2)
        V= savgol_filter(V, window_length=11, polyorder=2)

        f1 = interpolate.interp1d(V, Q, kind='linear')
        num_points = int((V[-1] - voltage_range) / step_len) + 1
        #num_points = int(len(V))
        V_new = np.linspace(V[0], V[-1], num_points)
        Q_new = f1(V_new)
        dQ = np.diff(Q_new)
        dV = np.diff(V_new)
        IC = dQ/dV
        return IC,V_new[1:]


In [ ]:
battery = Battery(cell_number)


In [ ]:
fig, ax = plt.subplots(figsize=(3, 2),dpi=200)
cm = plt.cm.GnBu(np.linspace(0,1,battery.cycle_life))
sm = plt.cm.ScalarMappable(cmap='GnBu',norm=plt.Normalize(vmin=0,vmax=battery.cycle_life))
for cycle in range(2,battery.cycle_life+1,5):
    IC,V = battery.get_IC_curve(cycle=cycle,voltage_range=3.6,step_len=0.01)
    plt.plot(V,IC,color=cm[cycle-1])
plt.xlabel('Voltage (V)')
plt.ylabel('dQ/dV (mAh/V)')
cbar = plt.colorbar(sm,cax=None,use_gridspec=True,ax=ax)
cbar.set_label('Cycle')
plt.show()

In [ ]:
fig, ax = plt.subplots(figsize=(3, 2),dpi=200)
cm = plt.cm.GnBu(np.linspace(0,1,battery.cycle_life))
sm = plt.cm.ScalarMappable(cmap='GnBu',norm=plt.Normalize(vmin=0,vmax=battery.cycle_life))
for cycle in range(1,battery.cycle_life+1,5): # 每隔5个cycle画一次
    voltage = battery.get_CC_value(cycle=cycle,variable='v',voltage_range=3.8)
    plt.plot(voltage,color=cm[cycle-1])
plt.xlabel('Time (s)')
plt.ylabel('Voltage (V)')
cbar = plt.colorbar(sm,ax=ax,use_gridspec=True)
cbar.set_label('Cycle')
plt.show()

In [ ]:
SOHs = []
start_cycle = 0
for cycle in range(start_cycle,battery.cycle_life):
    SOHs.append(battery.get_SOH(cycle=cycle))

SOHs = SOHs[start_cycle:]
plt.figure(figsize=(3, 2),dpi=200)
plt.plot(range(start_cycle,battery.cycle_life),SOHs,label='SOH')
plt.xlabel('Cycle')
plt.ylabel('SOH')
plt.show()

degradation_start_cycle = np.zeros(len(SOHs), dtype=bool)
k_start = np.argmax(SOHs)
degradation_start_cycle[k_start:] = True
